# 🏠 House Price Prediction — Linear Regression

**Dataset:** Ames Housing (publicly available)  
**Goal:** Predict residential property prices using multiple linear regression  
**Topics:** EDA, Feature Engineering, Multicollinearity (VIF), RMSE, R²

---

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded!')

## 2. Load Dataset

Using the Ames Housing dataset — publicly available on OpenML and Kaggle.

In [ ]:
from sklearn.datasets import fetch_openml
ames = fetch_openml(name='house_prices', as_frame=True, version=1)
df = ames.frame
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(f'Missing values (top 10):\n{df.isnull().sum().sort_values(ascending=False).head(10)}')
print(f'\nShape: {df.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['SalePrice'].astype(float), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice Distribution')
axes[1].hist(np.log1p(df['SalePrice'].astype(float)), bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Log(SalePrice) Distribution')
plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
top_corr = corr[['SalePrice']].sort_values('SalePrice', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(top_corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Top Features Correlated with SalePrice')
plt.tight_layout()
plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
features = ['GrLivArea', 'OverallQual', 'GarageCars', 'TotalBsmtSF',
            '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd']
target = 'SalePrice'

df_model = df[features + [target]].dropna().astype(float)
X = df_model[features]
y = np.log1p(df_model[target])
print(f'X shape: {X.shape}, y shape: {y.shape}')

## 5. Multicollinearity Check (VIF)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

vif_df = pd.DataFrame({'Feature': features,
    'VIF': [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
})
print(vif_df.sort_values('VIF', ascending=False).to_string(index=False))
print('\nVIF > 10 = high multicollinearity')

## 6. Model Training & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f'RMSE (log scale): {rmse:.4f}')
print(f'R² Score:         {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual log(SalePrice)')
axes[0].set_ylabel('Predicted log(SalePrice)')
axes[0].set_title(f'Actual vs Predicted  |  R² = {r2:.4f}')

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='coral')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient')
colors = ['seagreen' if c > 0 else 'tomato' for c in coef_df['Coefficient']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Feature Coefficients — Linear Regression')
plt.tight_layout()
plt.show()

## 7. Summary

| Metric | Value |
|--------|-------|
| R² Score | ~0.82 |
| RMSE (log scale) | ~0.17 |

**Key Findings:**
- `OverallQual` and `GrLivArea` are the strongest predictors
- Log-transforming the target normalizes the distribution
- VIF confirms acceptable multicollinearity

**Next Steps:** Try Ridge/Lasso regularization → see `regularization-deep-dive` repo.